# Sympy: Laplace Transform

*Notebook version of the appendix [Sympy: Laplace Transform](https://seoultechpse.github.io/EngMath/x_sympy_Laplace_transform.html) in the [Engineering Mathematics](https://seoultechpse.github.io/EngMath/) notes. It is generated from `x_sympy_Laplace_transform.qmd`; edits to this file are overwritten.*

In [ ]:
# Colab setup: install what the runtime does not provide
import sys
if 'google.colab' in sys.modules:
    %pip install -q "sympy>=1.14"

**Used in the main text:** [Definition of the Laplace Transform](https://seoultechpse.github.io/EngMath/ch_04_The_Laplace_Transform.html#sec-4-1), [The Laplace Transform](https://seoultechpse.github.io/EngMath/ch_x1_Parabolic_PDEs.html#sec-x1-15)

## Importing sympy

In [ ]:
import sympy
from sympy import (laplace_transform, inverse_laplace_transform,
                   laplace_correspondence, laplace_initial_conds, pi)
sympy.init_printing()

from IPython.display import display

print("sympy: ", sympy.__version__)

* In recent versions of sympy (1.13 and later), $~$`sympy.laplace_transform` can transform **derivatives of unknown functions**, $~$apply both **translation theorems**, $~$the **scaling** rule $f(at)$, $~$and the rule for $t^n f(t)$. $~$Two companion functions finish the job:

  * `laplace_correspondence(expr, {f: F})` $~$replaces $\mathcal{L}\{f(t)\}$ by $F(s)$

  * `laplace_initial_conds(expr, t, {y: [y0, y1, ...]})` $~$substitutes the initial values $y(0)$, $y'(0)$, $\dots$

* A few rules are still missing in sympy 1.14: $~$integrals $\int_0^t$, $~$convolution integrals, $~$and products such as $e^{ct}\, t^n f(t)$. $~$For these we use a small wrapper [_laplace_transform_expansion.py](https://github.com/SeoulTechPSE/EngMath/blob/main/_laplace_transform_expansion.py), $~$whose function `laplace_transform_()` adds only those rules and otherwise calls `sympy.laplace_transform`

In [ ]:
# Process Systems Lab., SeoulTech
# April 15, 2021 Authored by Kee-Youn Yoo
# April 4, 2024 Modified for sympy version 1.12
# November 2026 Rewritten for sympy 1.14: sympy.laplace_transform now handles
#   derivatives, both translation theorems, scaling f(a t) and t**n f(t) itself,
#   and sympy.laplace_correspondence / sympy.laplace_initial_conds replace the
#   old subs_() helper. This wrapper only adds the rules sympy still lacks:
#     * integrals from 0 to t           L{int_0^t h dtau}               = L{h}/s
#     * convolution integrals            L{int_0^t f(tau) g(t-tau) dtau} = F G
#     * powers of t                      L{t**k h}  = (-1)**k d^k/ds^k L{h}
#     * exponential factors              L{exp(c t) h} = L{h}(s - c)
#   (sympy 1.14 raises "Can't calculate derivative wrt s + c" for exp(c t) t f(t)
#   and returns a garbled sum for t**m t**n f(t).)

import sympy


def _is_zero_to_t(limits, t):
    _, lo, hi = limits
    return lo == 0 and hi == t


def _convolution_factors(h, tau, t):
    """Split h into A(tau) * B(t - tau) if possible; return (A, B) or None."""
    u = sympy.Dummy('u')
    A, B = sympy.Integer(1), sympy.Integer(1)
    for fac in sympy.Mul.make_args(h):
        if t not in fac.free_symbols:
            A *= fac                       # depends on tau only
        elif tau not in fac.subs(t, tau + u).free_symbols:
            B *= fac                       # depends on t - tau only
        else:
            return None
    if A.has(tau) and B.has(t):
        return A, B
    return None


def laplace_transform_(expr, t, s):
    """Laplace transform of expr(t): sympy.laplace_transform plus the rules above."""
    expr = sympy.sympify(expr)

    if expr.is_Add:
        return sympy.Add(*[laplace_transform_(term, t, s) for term in expr.args])

    if isinstance(expr, sympy.Integral) and _is_zero_to_t(expr.limits[-1], t):
        tau = expr.limits[-1][0]
        if len(expr.limits) == 1:
            ab = _convolution_factors(expr.function, tau, t)
            if ab is not None:                                   # convolution theorem
                A, B = ab
                return (laplace_transform_(A.subs(tau, t), t, s)
                        * laplace_transform_(B.subs(t, t + tau), t, s))
            inner = expr.function
        else:                                                    # repeated integral
            inner = sympy.Integral(expr.function, *expr.limits[:-1])
        return laplace_transform_(inner.subs(tau, t), t, s) / s

    if expr.is_Mul:
        const, rest = expr.as_independent(t)
        if const != 1:
            return const * laplace_transform_(rest, t, s)
        rest = sympy.powsimp(rest)                               # t**m * t**n -> t**(m + n)
        # exp(c t) first: differentiating LaplaceTransform(f, t, s + c) with respect to s
        # is evaluated incorrectly (as 0) by sympy, so the shift must be applied last
        for fac in sympy.Mul.make_args(rest):
            if isinstance(fac, sympy.exp):                       # first translation theorem
                c = sympy.cancel(fac.args[0] / t)
                if not c.has(t):                                 # transform in a dummy variable,
                    sigma = sympy.Dummy('sigma')                 # then shift: s -> s - c
                    return laplace_transform_(rest / fac, t, sigma).subs(sigma, s - c)
        for fac in sympy.Mul.make_args(rest):
            base, k = fac.as_base_exp()
            if base == t and not k.has(t) and (k.is_integer or k.is_Symbol):
                h = rest / fac                                   # derivative of the transform
                return (-1)**k * sympy.diff(laplace_transform_(h, t, s), (s, k))

    return sympy.laplace_transform(expr, t, s, noconds=True)

## Symbols and Functions

In [ ]:
a, b = sympy.symbols("a, b", positive=True)
n, m = sympy.symbols("n, m", positive=True, integer=True)

t, tau, tau1, tau2 = sympy.symbols("t, tau, tau1, tau2", positive=True)
s = sympy.symbols("s")

y, f, g = sympy.symbols("y, f, g", cls=sympy.Function)
Y, F, G = sympy.symbols("Y, F, G", cls=sympy.Function)

* A short helper: $~$transform and write $\mathcal{L}\{y\}$, $\mathcal{L}\{f\}$, $\mathcal{L}\{g\}$ as $Y(s)$, $F(s)$, $G(s)$

In [ ]:
def Lt(expr):
    return laplace_correspondence(laplace_transform_(expr, t, s), {y: Y, f: F, g: G})

## Laplace transform

In [ ]:
laplace_transform(1, t, s, noconds=True)

In [ ]:
laplace_transform(t, t, s, noconds=True)

In [ ]:
laplace_transform(sympy.exp(-3 *t), t, s, noconds=True)

In [ ]:
laplace_transform(2 *sympy.Heaviside(t -3), t, s, noconds=True)

In [ ]:
laplace_transform(sympy.sin(2 *t)**2, t, s, noconds=True)

## Inverse Laplace transform

In [ ]:
inverse_laplace_transform(1 / s**3, s, t)

In [ ]:
inverse_laplace_transform((-2 *s +6) / (s**2 + 4), s, t)

## Laplace transform of derivatives

In [ ]:
eq = y(t).diff(t, 2) +2 *y(t).diff(t) +10 *y(t)
eq

In [ ]:
Leq = laplace_correspondence(laplace_transform(eq, t, s, noconds=True), {y: Y})
Leq

In [ ]:
Leq = laplace_initial_conds(Leq, t, {y: [1, 1]})   # y(0) = 1, y'(0) = 1
Leq

In [ ]:
sol = sympy.solve(Leq, Y(s))[0]
sol

## Laplace transform of integrals

In [ ]:
eq = sympy.integrate(g(tau1), (tau1, 0, t)) +sympy.integrate(sympy.integrate(f(tau2), (tau2, 0, tau1)), (tau1, 0, t))
eq

* `sympy.laplace_transform` leaves these integrals unevaluated:

In [ ]:
laplace_transform(eq, t, s, noconds=True)

* The wrapper uses $~\mathcal{L}\left\{\int_0^t h(\tau)\,d\tau\right\} = \frac{1}{s}\mathcal{L}\{h(t)\}$:

In [ ]:
Lt(eq)

## Convolution

In [ ]:
eq = sympy.integrate(f(tau)*g(t -tau), (tau, 0, t)) +sympy.integrate(f(t -tau)*g(tau), (tau, 0, t))
eq

In [ ]:
Lt(eq)

In [ ]:
Lt(sympy.integrate(sympy.exp(tau) *sympy.sin(t -tau), (tau, 0, t)))  # e^t * sin t

**Used in the main text:** [Additional Operational Properties](https://seoultechpse.github.io/EngMath/ch_04_The_Laplace_Transform.html#sec-4-4)

## First translation theorem

In [ ]:
eq = y(t) *sympy.exp(-a *t)
eq

In [ ]:
Lt(eq)

## Second translation theorem

In [ ]:
eq = f(t -a) *sympy.Heaviside(t -a)
eq

In [ ]:
Lt(eq)

In [ ]:
eq = g(t) *sympy.Heaviside(t -a)
eq

In [ ]:
Lt(eq)   # alternative form: exp(-a s) L{g(t + a)}

In [ ]:
eq = sympy.cos(t) *sympy.Heaviside(t -pi)
eq

In [ ]:
Lt(eq)

## Derivatives of Transforms

In [ ]:
eq = 3 *f(t) *t**m *t**n
eq

In [ ]:
Lt(eq)

In [ ]:
eq =  t**n *t**m *f(t)*sympy.exp(-2*t)
eq

In [ ]:
Lt(eq)

---

In [ ]:
eq = t**n *sympy.diff(f(t), t, t)
eq

In [ ]:
Lt(eq)

In [ ]:
eq = t *sympy.integrate(f(tau), (tau, 0, t))
eq

In [ ]:
Lt(eq)

---

In [ ]:
eq = t *f(t) *sympy.exp(-t)
eq

In [ ]:
Lt(eq)

* For this product, $~$`sympy.laplace_transform` itself (1.14) raises an error:

In [ ]:
try:
    laplace_transform(eq, t, s)
except ValueError as err:
    print("ValueError:", err)

---

In [ ]:
eq = t *sympy.diff(f(t), t) *sympy.exp(-4*t)
eq

In [ ]:
Lt(eq)

**Used in the main text:** [Additional Operational Properties](https://seoultechpse.github.io/EngMath/ch_04_The_Laplace_Transform.html#sec-4-4)

## Various Transform Examples

In [ ]:
eq = sympy.exp(-a*t) *sympy.diff(f(t), t, t)
eq

In [ ]:
Lt(eq)

---

In [ ]:
eq = sympy.exp(-4*t) *sympy.integrate(f(tau), (tau, 0, t))
eq

In [ ]:
Lt(eq)

---

In [ ]:
eq = f(3*a*t)
eq

In [ ]:
Lt(eq)   # scaling: F(s/k)/k with k = 3a

$~$

## **Example 1**

A technique that can be used to solve certain ODE problems is to Laplace transform the ODE, which for many problems results in an algebraic equation that is easier to solve. The solution to the algebraic equation can then be transformed back to the original domain with an inverse Laplace transform, to obtain the solution to the original problem. For example, consider the following differential equation for a driven harmonic oscillator:

$$ \frac{d^2 y}{dt^2} + 2\frac{dy}{dt} +10 y = 2\sin 3t $$

$~$

In [ ]:
t = sympy.symbols('t', positive=True)
s = sympy.symbols('s')
y, Y = sympy.symbols('y, Y', cls=sympy.Function)

In [ ]:
ode = y(t).diff(t, 2) +2 *y(t).diff(t) +10 *y(t) -2 *sympy.sin(3*t)
ode

In [ ]:
Lode = laplace_correspondence(laplace_transform(ode, t, s, noconds=True), {y: Y})
Lode

* At this point, we need to specify the initial conditions for the ODE problem. Here we use
$y(0)=1$ and $y'(0)=0$, and substitute them into the Laplace-transformed ODE with `laplace_initial_conds`:

In [ ]:
Lode = laplace_initial_conds(Lode, t, {y: [1, 0]})
Lode

* This is an algebraic equation that can be solved for $Y(s)$

In [ ]:
Ysol = sympy.solve(Lode, Y(s))
Ysol[0]

* The result is a list of solutions, which in this case contains only one element. Performing the inverse Laplace transformation on this expression gives the solution to the original problem in the time domain (older versions of sympy needed the partial fractions to be inverted one by one):

In [ ]:
ysol = inverse_laplace_transform(Ysol[0], s, t)
ysol.expand().collect([sympy.exp(-t)])

* The same result is obtained with `dsolve`:

In [ ]:
sympy.dsolve(ode, y(t), ics={y(0): 1, y(t).diff(t).subs(t, 0): 0})

**Used in the main text:** [The Inverse Transform and Transforms of Derivatives](https://seoultechpse.github.io/EngMath/ch_04_The_Laplace_Transform.html#sec-4-2)

## **Example 2**

In some instances, the Laplace transform can be used to solve linear differential equations with variable monomial coefficients

$$ty'' + y' +ty =0, \;\;y(0)=1, \;y'(0)=0$$

$~$

In [ ]:
s, t = sympy.symbols('s, t', positive=True)
y, Y = sympy.symbols('y, Y', cls=sympy.Function)

ode = t *y(t).diff(t, 2) +y(t).diff(t) +t *y(t)
ode

$~$

* Reduce the given differential equation to a linear first-order DE in the transformed function $Y(s)=\mathcal{L}_t\left[y(t)\right]$. $~$Since $\mathcal{L}\{t\,h(t)\} = -\frac{d}{ds}\mathcal{L}\{h(t)\}$, $~$the factor $t$ becomes a derivative with respect to $s$

In [ ]:
Lode = laplace_transform(ode, t, s, noconds=True)
Lode = laplace_initial_conds(laplace_correspondence(Lode, {y: Y}), t, {y: [1, 0]})
Lode = Lode.doit().expand().collect(Y(s).diff(s))
Lode

* Solve the first-order ODE for $Y(s)$ and then find $y(t)=\mathcal{L}_t^{-1} \left[Y(s) \right]$

In [ ]:
sol = sympy.dsolve(Lode, Y(s))
sol

In [ ]:
y = inverse_laplace_transform(sol.rhs, s, t)
y

* The constant follows from $y(0)=1$ $~$(note $J_0(0)=1$)

In [ ]:
C1 = sympy.solve(sympy.Eq(y.subs(t, 0), 1))[0]
y.subs(sympy.Symbol('C1'), C1)

**Used in the main text:** [Additional Operational Properties](https://seoultechpse.github.io/EngMath/ch_04_The_Laplace_Transform.html#sec-4-4)